# Compute METs (SQL)

This notebook **does not compute METs in Python**. It picks **one dataset in a project** (projects may have several) and calls the W4H API:

1. `GET /projects/:id` — list linked datasets
2. `GET /datasets/:id/signals` — table / mapping metadata
3. `POST /datasets/:id/contributions/w4h-sql-compute-mets/sql` — API binds `sql/compute.sql` and runs it in Postgres

Formula (in the SQL file, interval-scaled): `METs ≈ calories / (weight_kg × 0.25)` per a 15-minute reference. **Not medical advice.**

Setup:

1. Import calories + weight into **your team** dataset (ICDE demo or Fitbit sample), not the read-only Samples sandbox if you want write-back
2. `cp .env.example .env` — set `W4H_API_KEY` and `W4H_PROJECT_ID`. Set `W4H_DATASET_ID` when the project has more than one dataset
3. Run the cells below, or: `w4h-sql-compute-mets datasets|signals|compute`

In [ ]:
try:
    import w4h_sql_compute_mets  # noqa: F401
    print("w4h_sql_compute_mets already installed.")
except ImportError:
    import subprocess
    import sys
    from pathlib import Path

    root = Path.cwd()
    if not (root / "pyproject.toml").is_file():
        root = (root / "..").resolve()
    print("kernel:", sys.executable)
    print("package:", root)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(root)])

In [ ]:
from w4h_sql_compute_mets.client import W4HClient
from w4h_sql_compute_mets.config import dataset_id, project_id
from w4h_sql_compute_mets.run import format_dataset_list, resolve_dataset_id, run_on_dataset

api = W4HClient()
pid = project_id() or None
did = dataset_id() or None
print("project:", pid or "(set W4H_PROJECT_ID to list datasets)")
print("dataset env:", did or "(unset — will require a pick if the project has several)")

if pid:
    linked = api.list_project_datasets(pid)
    print("linked datasets:")
    print(format_dataset_list(linked))
else:
    linked = []

In [ ]:
# Always one dataset. If the project has several, set DATASET_ID (or W4H_DATASET_ID) explicitly.
DATASET_ID = did  # or paste a catalog id from the list above

target_id, _ = resolve_dataset_id(dataset_id=DATASET_ID, project_id=pid, client=api)
print("using dataset:", target_id)

signals = api.get_signals(target_id)
for row in signals.get("signals") or []:
    print(
        f"{row.get('signal_type') or row.get('name')}: "
        f"{row.get('physical_schema')}.{row.get('physical_table')} "
        f"interval={row.get('interval_minutes')} prefix={row.get('table_prefix')}"
    )

In [ ]:
# Preview: API binds sql/compute.sql. Pass interval_minutes if signals above show interval=None
# (5 for ICDE demo, 1 for Fitbit).
INTERVAL_MINUTES = None

result = run_on_dataset(
    target_id,
    project_id=pid,
    write_back=False,
    interval_minutes=INTERVAL_MINUTES,
    limit=20,
    client=api,
)
run = result["run"]
print("columns:", run.get("columns"))
print("row_count:", run.get("row_count"), "interval:", run.get("interval_minutes"))
for row in (run.get("rows") or [])[:10]:
    print(row)

In [ ]:
# Admin only. Uncomment to materialize {prefix}_mets_computed on this dataset.
# written = run_on_dataset(
#     target_id,
#     project_id=pid,
#     write_back=True,
#     interval_minutes=INTERVAL_MINUTES,
#     client=api,
# )
# print(written["run"])